# 🚀 Lunar Lander with Reinforcement Learning (DQN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/07_reinforcement_learning/n16_lunar_lander_reinforcement_learning.ipynb)

> **Tiempo / time:** a CPU is enough (the network is tiny; a GPU does not help here). With the default settings training takes
> about **2–4 min** on a laptop or Colab CPU (`QUICK_RUN = True`: < 1 min, but the agent will not land yet).
> A **pre-trained network** is provided, so you can always see a good landing even if your own training is short.

This notebook demonstrates reinforcement learning using the classic Lunar Lander environment from Gymnasium (formerly OpenAI Gym). We implement a Deep Q-Network (DQN) and train an agent to land a spacecraft safely.

## Learning Objectives
- Understand the Lunar Lander environment
- Implement the Deep Q-Network (DQN) algorithm
- Train an agent to land successfully
- Visualize the agent **before and after** training and analyse the training process
- Compare your own network with a pre-trained one and different hyperparameters

## Environment Description
The Lunar Lander environment simulates landing a spacecraft on the moon. The agent must control the lander's engines to achieve a safe landing while minimizing fuel consumption.

**State Space**: 8 continuous values representing position, velocity, angle, and contact sensors
**Action Space**: 4 discrete actions (do nothing, fire left engine, fire main engine, fire right engine)
**Reward**: +100 for landing safely, -100 for crashing, fuel penalty for engine use. An episode with a total score ≥ 200 counts as a good landing.

🏭 **Industrial link**: this is a nonlinear control problem with saturated, on/off actuators (thrusters) — think of positioning a crane load
or a drone. In n19 we compare a DQN agent with classical controllers (PID, LQR) on a cart-pole.

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
    !apt-get -qq install -y swig > /dev/null
    %pip install -q "gymnasium[box2d]"
print('Running in Colab' if IN_COLAB else 'Running locally')

## 1. Environment Setup and Installation

Lunar Lander needs the **Box2D** physics engine (`pip install "gymnasium[box2d]"`). If it cannot be installed, the next cell prints
instructions and the notebook shows a **pre-computed animation** of the pre-trained agent instead of failing.

In [ ]:
# Import required libraries
import os
import io
import time
import random
import urllib.request
import warnings
from pathlib import Path
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, Image, display
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

warnings.filterwarnings('ignore', message='Animation was deleted')
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
HEADLESS = os.environ.get('HEADLESS', '0') == '1'      # no display (e.g. nbconvert): no widgets
USE_WIDGETS = HAS_WIDGETS and not HEADLESS

# ---- Configuration -----------------------------------------------------------------------------
QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'    # True = quick test (the agent will not learn to land)
N_EPISODES = 150 if QUICK_RUN else 700                 # training episodes (stops earlier if the average reaches 200)
SAVE_GIFS = True
FIG_DIR = Path('./figures_lunar'); FIG_DIR.mkdir(exist_ok=True)

# Pre-trained weights: first a local copy (next to this notebook), otherwise downloaded from GitHub (main branch)
REPO_RAW = 'https://raw.githubusercontent.com/samtzai/course_ai_for_industrial_control_students/main/notebooks/07_reinforcement_learning'
PRETRAINED_FILE = Path('weights/lunar_lander_dqn_pretrained.pt')
PRETRAINED_GIF = Path('weights/lunar_lander_pretrained.gif')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')   # a 9k-parameter network trains faster on CPU than on GPU (no transfer overhead)

plt.rcParams.update({'figure.facecolor': 'white', 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#eef2f7', 'font.size': 9, 'axes.titleweight': 'bold',
                     'animation.embed_limit': 60, 'figure.dpi': 80})

# ---- Gymnasium + Box2D ---------------------------------------------------------------------------
BOX2D_OK = False
try:
    if os.environ.get('SIMULATE_NO_BOX2D'):   # only for testing the fallback path
        raise ImportError('Box2D disabled by SIMULATE_NO_BOX2D')
    import gymnasium as gym
    _test_env = gym.make('LunarLander-v3', render_mode='rgb_array'); _test_env.reset(seed=0); _test_env.close()
    BOX2D_OK = True
    print(f'✓ Gymnasium {gym.__version__} with Box2D is working')
except Exception as e:
    print(f'❌ LunarLander could not be created: {type(e).__name__}: {e}')
    print("""
How to fix it:
  • Colab: run the first cell again (it installs swig and gymnasium[box2d]) and restart the runtime.
  • Linux / macOS:  pip install swig  &&  pip install "gymnasium[box2d]"
  • Windows: pip install swig, then pip install "gymnasium[box2d]". If box2d-py fails to build, install the
    "Microsoft C++ Build Tools" or use a conda environment:  conda install -c conda-forge box2d-py
Meanwhile the notebook skips the cells that need the simulator and shows the pre-computed animation.""")

print(f'PyTorch {torch.__version__} | NumPy {np.__version__} | widgets: {USE_WIDGETS} | QUICK_RUN: {QUICK_RUN} | episodes: {N_EPISODES}')


def fetch(local_path, url):
    """Return the bytes of a file: local copy if present, otherwise download it (None if not possible)."""
    for p in [Path(local_path), Path(local_path).name and Path(Path(local_path).name)]:
        if p.exists():
            return p.read_bytes()
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            data = r.read()
        Path(local_path).parent.mkdir(parents=True, exist_ok=True); Path(local_path).write_bytes(data)
        print(f'⬇️  downloaded {url}')
        return data
    except Exception as e:
        print(f'⚠️  could not get {local_path} ({type(e).__name__})')
        return None


if not BOX2D_OK:
    gif = fetch(PRETRAINED_GIF, f'{REPO_RAW}/{PRETRAINED_GIF.as_posix()}')
    if gif:
        print('Pre-computed landing of the pre-trained agent:')
        display(Image(data=gif, format='gif'))

## 2. Exploring the Lunar Lander Environment

In [ ]:
# Create and explore the environment
action_meanings = {0: "Do nothing", 1: "Fire left orientation engine", 2: "Fire main engine", 3: "Fire right orientation engine"}
obs_descriptions = ["Horizontal coordinate", "Vertical coordinate", "Horizontal velocity", "Vertical velocity",
                    "Angle", "Angular velocity", "Left leg contact (boolean)", "Right leg contact (boolean)"]
state_size, action_size = 8, 4

if BOX2D_OK:
    env = gym.make('LunarLander-v3', render_mode='rgb_array')
    print("✓ LunarLander environment created successfully!")
    print("=== Lunar Lander Environment Information ===")
    print(f"Observation space: {env.observation_space}")
    print(f"Action space: {env.action_space}")
    print(f"Number of actions: {env.action_space.n}")
    state_size, action_size = env.observation_space.shape[0], env.action_space.n
print("\n=== Action Space ===")
for action, meaning in action_meanings.items():
    print(f"Action {action}: {meaning}")
print("\n=== Observation Space ===")
for i, desc in enumerate(obs_descriptions):
    print(f"Index {i}: {desc}")

### Before training: a random agent

`display_episode` turns a list of frames into an animation (works in Colab and VS Code); it can also save it as a GIF.

In [ ]:
def display_episode(frames, title, gif=None, every=2, as_gif=False):
    """Display an episode as an embedded animation (and optionally save it as a GIF)."""
    frames = frames[::every]
    fig, ax = plt.subplots(figsize=(4, 3), dpi=72)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    image = ax.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return (image,)

    fps = 50 / every
    episode_animation = animation.FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)
    plt.close(fig)
    if SAVE_GIFS and gif:
        episode_animation.save(FIG_DIR / gif, writer=animation.PillowWriter(fps=fps / 2), dpi=60)
        print(f'GIF saved to {FIG_DIR / gif}')
    if as_gif:
        tmp = FIG_DIR / '_tmp.gif'; episode_animation.save(tmp, writer=animation.PillowWriter(fps=fps / 2), dpi=60)
        display(Image(data=tmp.read_bytes(), format='gif')); tmp.unlink(missing_ok=True)
    else:
        display(HTML(episode_animation.to_jshtml(default_mode='once')))


def run_episode(env, policy, seed=None, render=True, max_steps=1000):
    """Run one episode with policy(state) -> action. Returns (score, steps, frames, actions)."""
    state, _ = env.reset(seed=seed)
    total, frames, actions = 0.0, [], []
    for step in range(max_steps):
        action = policy(state)
        state, reward, terminated, truncated, _ = env.step(action)
        total += reward; actions.append(action)
        if render:
            frames.append(env.render())
        if terminated or truncated:
            break
    return total, step + 1, frames, actions


if BOX2D_OK:
    print("Running random episode...")
    score, steps, frames, _ = run_episode(env, lambda s: env.action_space.sample(), seed=7)
    print(f"Episode ended after {steps} steps | Total reward: {score:.2f}")
    display_episode(frames, f'BEFORE: random agent | score {score:.0f}', gif='n16_antes_aleatorio.gif')

## 3. Deep Q-Network (DQN) Implementation

Now we'll implement a Deep Q-Network to learn how to land the lunar lander. DQN uses a neural network to approximate the Q-function, which estimates the expected future reward for each action in a given state.

Two details that matter in practice (and that were simplified in older versions of this notebook):
- The replay buffer stores **`terminated`** (crash or landing) as the end of the episode, **not** `truncated` (time limit): after a time-out the future still has value, so we keep bootstrapping.
- We use the **Huber loss** and gradient clipping, which make the training much more stable than plain MSE.

In [ ]:
# Experience replay buffer
Experience = namedtuple('Experience', ('state', 'action', 'reward', 'next_state', 'done'))


class ReplayBuffer:
    """Experience replay buffer for DQN"""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Add experience to buffer"""
        self.buffer.append(Experience(state, action, reward, next_state, done))

    def sample(self, batch_size):
        """Sample random batch of experiences"""
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

print("Experience replay buffer implemented!")

In [ ]:
# Deep Q-Network architecture
class DQN(nn.Module):
    """Deep Q-Network for Lunar Lander"""

    def __init__(self, state_size, action_size, hidden_size=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, action_size))

    def forward(self, x):
        return self.network(x)


test_dqn = DQN(state_size, action_size)
print("DQN Architecture:")
print(test_dqn)
print(f"\nTotal parameters: {sum(p.numel() for p in test_dqn.parameters() if p.requires_grad):,}")

In [ ]:
# DQN Agent implementation
class DQNAgent:
    """DQN Agent for Lunar Lander (double=True gives Double DQN, see section 8)"""

    def __init__(self, state_size, action_size, lr=0.0005, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
                 buffer_size=100000, batch_size=64, update_freq=4, target_update_freq=1000, double=True):
        self.state_size, self.action_size = state_size, action_size
        self.lr, self.gamma = lr, gamma
        self.epsilon, self.epsilon_end, self.epsilon_decay = epsilon_start, epsilon_end, epsilon_decay
        self.batch_size, self.update_freq, self.target_update_freq = batch_size, update_freq, target_update_freq
        self.double = double

        # Neural networks
        self.device = device
        self.q_network = DQN(state_size, action_size).to(self.device)
        self.target_network = DQN(state_size, action_size).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)

        # Experience replay
        self.replay_buffer = ReplayBuffer(buffer_size)
        self.step_count = 0

    @torch.no_grad()
    def act(self, state, training=True):
        """Choose action using epsilon-greedy policy"""
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_size)
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        return self.q_network(state_tensor).argmax().item()

    def store_experience(self, state, action, reward, next_state, done):
        """Store experience in replay buffer"""
        self.replay_buffer.push(state, action, reward, next_state, done)

    def learn(self):
        """Train the network on a batch of experiences"""
        if len(self.replay_buffer) < self.batch_size:
            return None
        batch = Experience(*zip(*self.replay_buffer.sample(self.batch_size)))
        states = torch.as_tensor(np.array(batch.state), dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(batch.action, dtype=torch.int64, device=self.device)
        rewards = torch.as_tensor(batch.reward, dtype=torch.float32, device=self.device)
        next_states = torch.as_tensor(np.array(batch.next_state), dtype=torch.float32, device=self.device)
        dones = torch.as_tensor(batch.done, dtype=torch.float32, device=self.device)

        # Current Q values
        current_q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values
        with torch.no_grad():
            if self.double:   # Double DQN: main network selects, target network evaluates
                next_actions = self.q_network(next_states).argmax(1, keepdim=True)
                next_q_values = self.target_network(next_states).gather(1, next_actions).squeeze(1)
            else:             # standard DQN
                next_q_values = self.target_network(next_states).max(1)[0]
            target_q_values = rewards + self.gamma * next_q_values * (1 - dones)

        # Compute loss (Huber) and optimize
        loss = F.smooth_l1_loss(current_q_values, target_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), 10)
        self.optimizer.step()
        return loss.item()

    def update_target_network(self):
        """Copy weights from main network to target network"""
        self.target_network.load_state_dict(self.q_network.state_dict())

    def decay_epsilon(self):
        """Decay epsilon for exploration"""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

print("DQN Agent implemented successfully!")

## 4. Training the Agent

The agent trains for at most `N_EPISODES` episodes and stops as soon as the average of the last 100 episodes reaches **200** (the usual
"solved" criterion). Every 50 episodes we also evaluate the greedy policy (no exploration) on 5 fixed episodes and keep the best network.

In [ ]:
def evaluate_policy(env, agent, n_episodes=5, seed=1000):
    return float(np.mean([run_episode(env, lambda s: agent.act(s, training=False), seed=seed + i, render=False)[0]
                          for i in range(n_episodes)]))


def train_dqn_agent(env, agent, n_episodes=N_EPISODES, max_steps=1000, target_score=200, print_every=50, save_model=True,
                    eval_env=None):
    """Train DQN agent on Lunar Lander"""
    scores, losses, epsilons, evals = [], [], [], []
    recent_scores = deque(maxlen=100)
    best = (-np.inf, None)
    t0 = time.time()
    print(f"Starting training for up to {n_episodes} episodes (target average score: {target_score})...")

    for episode in tqdm(range(1, n_episodes + 1), desc="Training", mininterval=5):
        state, _ = env.reset()
        total_reward, episode_losses = 0.0, []
        for step in range(max_steps):
            action = agent.act(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            agent.store_experience(state, action, reward, next_state, terminated)   # only real ends are terminal
            if agent.step_count % agent.update_freq == 0:
                loss = agent.learn()
                if loss is not None:
                    episode_losses.append(loss)
            if agent.step_count % agent.target_update_freq == 0:
                agent.update_target_network()
            agent.step_count += 1
            state = next_state
            total_reward += reward
            if terminated or truncated:
                break

        agent.decay_epsilon()
        scores.append(total_reward); recent_scores.append(total_reward); epsilons.append(agent.epsilon)
        losses.append(np.mean(episode_losses) if episode_losses else 0.0)

        if episode % print_every == 0:
            avg_score = np.mean(recent_scores)
            ev = evaluate_policy(eval_env or env, agent)
            evals.append((episode, ev))
            if ev > best[0]:
                best = (ev, {k: v.clone() for k, v in agent.q_network.state_dict().items()})
            print(f"Episode {episode:4d} | Avg Score (100): {avg_score:7.2f} | Greedy eval: {ev:7.1f} | "
                  f"Epsilon: {agent.epsilon:.3f} | Loss: {losses[-1]:.3f} | {time.time() - t0:4.0f} s")
            if avg_score >= target_score:
                print(f"\n🎉 Target reached (checked at episode {episode}): average score over the last 100 episodes = {avg_score:.2f}")
                break

    train_time = time.time() - t0
    if best[1] is not None and best[0] > evaluate_policy(eval_env or env, agent):
        agent.q_network.load_state_dict(best[1])    # keep the best network seen during training
    if save_model:
        torch.save(agent.q_network.state_dict(), 'lunar_lander_dqn.pth')
        print("Model saved as 'lunar_lander_dqn.pth'")
    print(f"Training time: {train_time:.0f} s | {len(scores)} episodes | {agent.step_count:,} steps")
    return scores, losses, epsilons, evals, train_time

print("Training function ready!")

In [ ]:
# Initialize agent and start training
agent = DQNAgent(state_size=state_size, action_size=action_size, lr=0.0005, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995, buffer_size=100000,
                 batch_size=64, update_freq=4, target_update_freq=1000, double=True)

if BOX2D_OK:
    eval_env = gym.make('LunarLander-v3')
    scores, losses, epsilons, evals, train_time = train_dqn_agent(env, agent, n_episodes=N_EPISODES, eval_env=eval_env)
else:
    scores, losses, epsilons, evals, train_time = [], [], [], [], 0.0
    print('⏭️ Skipped: Box2D is not available.')

## 5. Training Analysis and Visualization

In [ ]:
# Plot training results
def plot_training_results(scores, losses, epsilons, evals):
    """Plot comprehensive training results"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 6))
    ax1.plot(scores, alpha=0.4, color='#2563eb', label='Episode score')
    window = min(100, max(1, len(scores) // 10))
    moving_avg = [np.mean(scores[max(0, i - window + 1):i + 1]) for i in range(len(scores))]
    ax1.plot(moving_avg, color='#dc2626', linewidth=2, label=f'Moving avg ({window})')
    if evals:
        ax1.plot(*zip(*evals), 'o-', color='#16a34a', ms=4, label='Greedy evaluation')
    ax1.axhline(y=200, color='#16a34a', linestyle='--', label='Target (200)')
    ax1.set(xlabel='Episode', ylabel='Score', title='Training scores'); ax1.legend(fontsize=7)
    ax2.plot(losses, color='#f59e0b', alpha=0.8); ax2.set(xlabel='Episode', ylabel='Huber loss', title='Training loss')
    ax3.plot(epsilons, color='#7c3aed'); ax3.set(xlabel='Episode', ylabel='Epsilon', title='Exploration rate (epsilon)')
    ax4.hist(scores, bins=40, alpha=0.7, color='#93c5fd', edgecolor='white')
    ax4.axvline(np.mean(scores), color='#dc2626', linestyle='--', label=f'Mean: {np.mean(scores):.1f}')
    ax4.axvline(200, color='#16a34a', linestyle='--', label='Target: 200')
    ax4.set(xlabel='Score', ylabel='Frequency', title='Score distribution'); ax4.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG_DIR / 'n16_training.png', dpi=100); plt.show()

    print("\n=== Training Summary ===")
    print(f"Total episodes: {len(scores)}")
    print(f"Average score: {np.mean(scores):.2f} | Best: {np.max(scores):.2f} | Worst: {np.min(scores):.2f}")
    print(f"Success rate (score >= 200): {np.mean(np.array(scores) >= 200) * 100:.1f}%")
    if len(scores) >= 100:
        print(f"Last 100 episodes average: {np.mean(scores[-100:]):.2f}")


if scores:
    plot_training_results(scores, losses, epsilons, evals)

## 6. Testing the Trained Agent — and the pre-trained one

We test **your** network and a **pre-trained** network (trained the same way for 1000 episodes, ≈ 2.5 min of CPU) on the same 10 episodes
(same random seeds, no exploration). The pre-trained weights are a 40 kB file in `weights/` next to this notebook; in Colab they are
downloaded from GitHub automatically.

In [ ]:
def load_pretrained_agent():
    data = fetch(PRETRAINED_FILE, f'{REPO_RAW}/{PRETRAINED_FILE.as_posix()}')
    if data is None:
        return None
    pre = DQNAgent(state_size, action_size)
    pre.q_network.load_state_dict(torch.load(io.BytesIO(data), map_location=device))
    pre.epsilon = 0.0
    return pre


def test_agent(env, agent, n_episodes=10, render=True, seed=2000, label='agent'):
    """Test an agent without exploration on fixed seeds"""
    test_scores, test_episodes = [], []
    print(f"Testing {label} for {n_episodes} episodes...")
    for episode in range(n_episodes):
        score, steps, frames, _ = run_episode(env, lambda s: agent.act(s, training=False), seed=seed + episode,
                                              render=render and episode < 2)
        test_scores.append(score)
        print(f"Episode {episode + 1}: Score = {score:.2f}, Steps = {steps}")
        if episode < 2:
            test_episodes.append({'episode': episode + 1, 'score': score, 'steps': steps, 'frames': frames})
    print(f"\n=== Test Results ({label}) ===")
    print(f"Average score: {np.mean(test_scores):.2f} ± {np.std(test_scores):.2f} | Best: {np.max(test_scores):.2f} | "
          f"Worst: {np.min(test_scores):.2f} | Success rate (>=200): {np.mean(np.array(test_scores) >= 200) * 100:.0f}%")
    return test_scores, test_episodes


pretrained = load_pretrained_agent()
if BOX2D_OK:
    test_scores, test_episodes = test_agent(env, agent, label='YOUR network')
    if pretrained is not None:
        pre_scores, pre_episodes = test_agent(env, pretrained, label='PRE-TRAINED network')

In [ ]:
# AFTER training: your network and the pre-trained one on the same episode
if BOX2D_OK:
    ep = test_episodes[0]
    display_episode(ep['frames'], f"AFTER: your network | score {ep['score']:.0f}", gif='n16_despues_tu_red.gif')
    if pretrained is not None:
        ep = pre_episodes[0]
        display_episode(ep['frames'], f"AFTER: pre-trained network | score {ep['score']:.0f}", gif='n16_despues_preentrenada.gif')

### 🎛️ Try it yourself

Choose the agent and the random seed of the episode (each seed is a different terrain and initial push) and watch the landing.
Without widgets, change the arguments of `pilot(...)`.

In [ ]:
def pilot(agent_name='pre-trained', seed=0):
    agents = {'random': None, 'your network': agent, 'pre-trained': pretrained}
    ag = agents[agent_name]
    if agent_name != 'random' and ag is None:
        print('That agent is not available.'); return
    policy = (lambda s: env.action_space.sample()) if ag is None else (lambda s: ag.act(s, training=False))
    score, steps, frames, actions = run_episode(env, policy, seed=int(seed))
    counts = np.bincount(actions, minlength=4)
    print(f'{agent_name} | seed {seed} | score {score:.1f} {"✅ good landing" if score >= 200 else ""} | {steps} steps | '
          + ', '.join(f'{action_meanings[a].split()[0] if a == 0 else action_meanings[a].split()[1]}: {c}' for a, c in enumerate(counts)))
    display_episode(frames, f'{agent_name} | seed {seed} | score {score:.0f}', every=3, as_gif=True)


if BOX2D_OK:
    if USE_WIDGETS:
        display(widgets.interactive(pilot, {'manual': True, 'manual_name': '▶ Play'},
                                    agent_name=widgets.ToggleButtons(options=['random', 'your network', 'pre-trained'], value='pre-trained'),
                                    seed=widgets.IntSlider(value=0, min=0, max=100, description='seed')))
    else:
        print('ipywidgets not available (or HEADLESS=1): static version.')
        pilot('pre-trained', seed=3)

## 7. Hyperparameter Analysis and Comparison

In [ ]:
# Compare different hyperparameters
def compare_hyperparameters(env, configs, n_episodes=300):
    """Compare different hyperparameter configurations"""
    results = {}
    for name, config in configs.items():
        print(f"\n=== Testing configuration: {name} === {config}")
        ag = DQNAgent(state_size=state_size, action_size=action_size, **config)
        sc, ls, eps, ev, tt = train_dqn_agent(env, ag, n_episodes=n_episodes, print_every=100, save_model=False)
        results[name] = {'scores': sc, 'avg_score': np.mean(sc[-100:]), 'success_rate': np.mean(np.array(sc[-100:]) >= 200) * 100,
                         'time_s': tt}
    return results


# Define configurations to compare
configs = {
    'Default': {'lr': 0.0005, 'gamma': 0.99, 'epsilon_decay': 0.995},
    'Higher LR': {'lr': 0.001, 'gamma': 0.99, 'epsilon_decay': 0.995},
    'Lower Gamma': {'lr': 0.0005, 'gamma': 0.95, 'epsilon_decay': 0.995},
    'Faster Decay': {'lr': 0.0005, 'gamma': 0.99, 'epsilon_decay': 0.99},
    'Standard DQN (no Double)': {'lr': 0.0005, 'gamma': 0.99, 'epsilon_decay': 0.995, 'double': False},
}

RUN_COMPARISON = False   # ✍️ set to True: about 1–1.5 min per configuration with 300 episodes on CPU
if RUN_COMPARISON and BOX2D_OK:
    results = compare_hyperparameters(env, configs, n_episodes=300)
    for name, r in results.items():
        plt.plot(np.convolve(r['scores'], np.ones(20) / 20, mode='valid'), label=f"{name} ({r['avg_score']:.0f})")
    plt.axhline(200, ls='--', color='#16a34a'); plt.xlabel('Episode'); plt.ylabel('Score (moving avg 20)'); plt.legend(fontsize=7); plt.show()
else:
    print("Hyperparameter comparison configurations ready! Set RUN_COMPARISON = True to run them.")

## 8. Advanced Techniques and Improvements

**Double DQN.** Standard DQN uses the *same* target network to choose and to evaluate the best next action ($\max_{a'} Q_{target}(s', a')$),
which overestimates Q-values. Double DQN lets the **main** network choose the action and the **target** network evaluate it.
In this notebook it is the `double=True` option of `DQNAgent.learn()`; the class below keeps the original name for reference.

In [ ]:
# Double DQN implementation (same as DQNAgent(double=True))
class DoubleDQNAgent(DQNAgent):
    """Double DQN Agent - reduces overestimation bias"""
    def __init__(self, *args, **kwargs):
        kwargs['double'] = True
        super().__init__(*args, **kwargs)

print("Double DQN agent implemented!")

In [ ]:
# Prioritized Experience Replay Buffer
class PrioritizedReplayBuffer:
    """Prioritized Experience Replay Buffer"""

    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.position = 0
        self.max_priority = 1.0

    def push(self, state, action, reward, next_state, done):
        """Add experience with maximum priority"""
        experience = Experience(state, action, reward, next_state, done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(experience)
        else:
            self.buffer[self.position] = experience
        self.priorities[self.position] = self.max_priority
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        """Sample batch with priorities"""
        prios = self.priorities if len(self.buffer) == self.capacity else self.priorities[:self.position]
        probs = prios ** self.alpha
        probs /= probs.sum()
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        weights = (len(self.buffer) * probs[indices]) ** (-beta)
        weights /= weights.max()
        return [self.buffer[idx] for idx in indices], indices, weights

    def update_priorities(self, indices, priorities):
        """Update priorities for sampled experiences"""
        for idx, priority in zip(indices, priorities):
            self.priorities[idx] = priority
            self.max_priority = max(self.max_priority, priority)

    def __len__(self):
        return len(self.buffer)

print("Prioritized Experience Replay implemented!")

## 9. Performance Analysis and Insights

In [ ]:
# Analyze agent performance
def analyze_agent_behavior(env, agent, n_episodes=5, label='agent'):
    """Analyze agent's decision making and behavior patterns"""
    print(f"=== Agent Behavior Analysis ({label}) ===")
    action_counts = {i: 0 for i in range(4)}
    data = []
    for episode in range(n_episodes):
        state, _ = env.reset(seed=3000 + episode)
        while True:
            action = agent.act(state, training=False)
            action_counts[action] += 1
            data.append((state.copy(), action))
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
    states = np.array([d[0] for d in data]); acts = np.array([d[1] for d in data])

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 6.5))
    names = ['Do Nothing', 'Left Engine', 'Main Engine', 'Right Engine']
    counts = [action_counts[a] for a in range(4)]
    bars = ax1.bar(names, counts, color=['#94a3b8', '#2563eb', '#dc2626', '#f59e0b'])
    for b, c in zip(bars, counts):
        ax1.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{c / sum(counts) * 100:.0f}%', ha='center', va='bottom', fontsize=8)
    ax1.set_title('Action usage distribution')
    sc = ax2.scatter(states[:, 0], states[:, 1], c=acts, cmap='viridis', s=8, alpha=0.6)
    ax2.set(xlabel='Horizontal position', ylabel='Vertical position', title='Actions vs position'); plt.colorbar(sc, ax=ax2, label='Action')
    sc2 = ax3.scatter(states[:, 2], states[:, 3], c=acts, cmap='viridis', s=8, alpha=0.6)
    ax3.set(xlabel='Horizontal velocity', ylabel='Vertical velocity', title='Actions vs velocity'); plt.colorbar(sc2, ax=ax3, label='Action')
    ax4.hist(states[:, 4], bins=20, alpha=0.7, color='#93c5fd', edgecolor='white')
    ax4.axvline(0, color='#dc2626', linestyle='--', label='Upright'); ax4.set(xlabel='Angle (rad)', title='Angle distribution'); ax4.legend()
    plt.tight_layout(); plt.show()

    total = sum(counts); best = max(action_counts, key=action_counts.get)
    print(f"Total actions analyzed: {total}")
    print(f"Most used action: {best} ({names[best]}, {action_counts[best] / total * 100:.1f}%)")
    print(f"Average angle: {np.mean(states[:, 4]):.3f} rad | Angle std: {np.std(states[:, 4]):.3f} rad")


if BOX2D_OK:
    analyze_agent_behavior(env, pretrained if pretrained is not None else agent, label='pre-trained' if pretrained is not None else 'your network')

## 10. Key Takeaways and Conclusion

### What We've Learned

1. **Environment Understanding**: The Lunar Lander environment is a challenging control problem with a shaped but delayed reward
2. **DQN Implementation**: We implemented a Deep Q-Network with:
   - Experience replay for stable learning
   - Target network for reduced correlation
   - Epsilon-greedy exploration strategy
   - Double DQN targets and the Huber loss
3. **Training Dynamics**:
   - Initial random exploration with high epsilon
   - Gradual improvement as the agent learns (a few hundred episodes on a CPU)
   - Importance of hyperparameter tuning and of storing only *real* terminal states
4. **Performance Analysis**: Successful agents learn to:
   - Control orientation with side engines
   - Use main engine for controlled descent
   - Balance fuel efficiency with safety

### Advanced Techniques Covered
- Double DQN for reduced overestimation
- Prioritized Experience Replay for better sample efficiency
- Comprehensive performance analysis

### Next Steps
- Try other RL algorithms (PPO, A3C, SAC)
- Experiment with different network architectures
- Compare RL with classical control on the cart-pole: **n19**
- Apply techniques to other environments

In [ ]:
# Save final model and create summary
def create_training_summary(scores, epsilons):
    """Create comprehensive training summary"""
    if not scores:
        print("No training run in this session (Box2D missing?) — see the pre-trained GIF above.")
        return None
    summary = {
        'total_episodes': len(scores),
        'final_average_score': np.mean(scores[-100:]) if len(scores) >= 100 else (np.mean(scores) if scores else float('nan')),
        'best_score': np.max(scores) if scores else float('nan'),
        'worst_score': np.min(scores) if scores else float('nan'),
        'success_rate': np.mean(np.array(scores) >= 200) * 100 if scores else float('nan'),
        'episodes_to_solve': None,
        'final_epsilon': epsilons[-1] if epsilons else None,
    }
    for i in range(100, len(scores) + 1):   # "solved" = average >= 200 over 100 consecutive episodes
        if np.mean(scores[i - 100:i]) >= 200:
            summary['episodes_to_solve'] = i
            break

    print("\n" + "=" * 50)
    print("           LUNAR LANDER TRAINING SUMMARY")
    print("=" * 50)
    print(f"📊 Total Episodes Trained: {summary['total_episodes']}  ({train_time:.0f} s)")
    print(f"🎯 Final Average Score: {summary['final_average_score']:.2f}")
    print(f"🏆 Best Score Achieved: {summary['best_score']:.2f}")
    print(f"💥 Worst Score: {summary['worst_score']:.2f}")
    print(f"✅ Success Rate (≥200): {summary['success_rate']:.1f}%")
    print(f"🎉 Avg-100 first reached ≥200 at episode {summary['episodes_to_solve']}" if summary['episodes_to_solve']
          else "❌ Environment not yet solved (need avg ≥200 for 100 episodes) — compare with the pre-trained network")
    if summary['final_epsilon']:
        print(f"🔍 Final Exploration Rate: {summary['final_epsilon']:.4f}")
    if BOX2D_OK:
        print(f"🧪 Test (10 episodes): your network {np.mean(test_scores):.1f}" +
              (f" | pre-trained {np.mean(pre_scores):.1f}" if pretrained is not None else ''))
    print("=" * 50)
    return summary


final_summary = create_training_summary(scores, epsilons)
if BOX2D_OK:
    env.close()
print("\n🚀 Lunar Lander RL training complete!")